# 04 · BERT como regresor (fine-tuning)

**Etapa 5.** Entorno nuevo de Colab con **GPU T4**.

- Modelo: `distilbert-base-uncased` (66 M parámetros; ~2× más rápido que BERT-base con ~97 % de su
  calidad), con `AutoModelForSequenceClassification(num_labels=1, problem_type="regression")`.
- Target escalado a [0, 1] (÷ tope); las predicciones se vuelven a años y se recortan a [0, tope].
- Pérdida MSE; Huber como alternativa en la Pregunta 3.
- Early stopping por MAE de validación (en años).

In [ ]:
!pip install -q "transformers==5.16.1" "accelerate==1.14.0" "scikit-learn==1.9.0"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/leruizdelavegat-max/HW_04_202602.git"
RAMA = "feature/pipeline"          # cambiar a "main" cuando se haga el merge del PR
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/HW_04_202602")
    if not (RAIZ / ".git").exists():
        subprocess.run(["git", "clone", "-b", RAMA, REPO_URL, str(RAIZ)], check=True)
    else:
        # Solo se actualiza el CÓDIGO: data/ y output/ de Drive no se tocan (un pull fallaría si ya existen)
        subprocess.run(["git", "-C", str(RAIZ), "fetch", "origin", RAMA], check=True)
        subprocess.run(["git", "-C", str(RAIZ), "checkout", f"origin/{RAMA}", "--",
                        "src", "app", "notebooks", "requirements.txt", "README.md"], check=True)
    version = subprocess.run(["git", "-C", str(RAIZ), "log", "-1", "--format=%h %s", f"origin/{RAMA}"],
                             capture_output=True, text=True).stdout.strip()
    print("Código de GitHub:", version)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import torch
except ImportError as e:
    raise RuntimeError("PyTorch no carga: este entorno tiene restos del notebook 01 (PaddlePaddle). Usa "
                       "«Entorno de ejecución → Desconectar y borrar entorno de ejecución» y vuelve a ejecutar.") from e
import transformers
from sklearn.metrics import mean_absolute_error
from transformers import AutoTokenizer

from src import config, modelos, texto

assert torch.cuda.is_available(), "Activa la GPU T4"
GPU = torch.cuda.get_device_name(0)
print("transformers", transformers.__version__, "· torch", torch.__version__, "·", GPU)

df = pd.read_parquet(config.PROCESADOS / "dataset.parquet")
TOPE = json.loads((config.PROCESADOS / "run_config.json").read_text())["tope"]
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
BASE = "distilbert-base-uncased"
MAX_LEN = 512
PRED = config.OUTPUT / "predicciones"; PRED.mkdir(exist_ok=True)

## ¿Cuántos CVs pasan de 512 tokens y qué se pierde al truncar?

Para cada estrategia se mide qué fracción de los **años** del texto sobrevive en lo que BERT realmente ve.
`cabeza_cola` = primeros 382 + últimos 128 tokens (el historial laboral suele ir arriba y la educación —con
su año de graduación— abajo).

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
largos = np.array([len(tok(t, add_special_tokens=False)["input_ids"]) for t in tr.texto_final])
print(f"Tokens por CV: mediana {np.median(largos):.0f} · p90 {np.percentile(largos, 90):.0f} · máx {largos.max()} · "
      f">510: {(largos > 510).mean():.1%}")

sobrevivencia = {}
for estr in ("cabeza", "cabeza_cola"):
    ids, n_trunc = modelos.codificar(tok, list(va.texto_final), MAX_LEN, estr)
    vistos = [modelos.texto_visto(tok, i) for i in ids]
    sobrevivencia[estr] = {"truncados_val": n_trunc,
                           "anios_que_sobreviven": float(np.nanmean([texto.recall_anios(a, b) for a, b in zip(va.texto_final, vistos)]))}
sobrevivencia = pd.DataFrame(sobrevivencia).T
sobrevivencia.to_csv(config.TABLAS / "truncamiento_sobrevivencia.csv")
sobrevivencia

In [ ]:
plt.figure(figsize=(7, 3.2))
plt.hist(largos, bins=40, color="#2a78d6"); plt.axvline(510, color="#d03b3b", ls="--", label="límite 510")
plt.title("Largo de los CVs en tokens (train)"); plt.legend(); plt.tight_layout()
plt.savefig(config.FIGURAS / "tokens_por_cv.png", dpi=130); plt.show()

## Entrenamiento principal

| hiperparámetro | valor |
|---|---|
| learning rate | 3e-5 (AdamW, decaimiento 0.01, warmup 6 %) |
| batch | 16 |
| épocas | máx. 8, early stopping con paciencia 2 sobre MAE val |
| longitud | 512 (cabeza + cola) |
| precisión | fp16 |
| semilla | 42 |

In [ ]:
HP = dict(modelo_base=BASE, max_len=MAX_LEN, estrategia="cabeza_cola", perdida="mse", lr=3e-5, batch=16,
          epocas=8, paciencia=2, seed=config.SEED)
SALIDA = config.MODELOS / "bert_regresor"
modelo, tok, seg_entrenamiento, historia = modelos.entrenar_bert(
    list(tr.texto_final), tr.y.values, list(va.texto_final), va.y.values, TOPE, SALIDA, **HP)
print(f"Entrenamiento: {seg_entrenamiento / 60:.1f} min en {GPU}")
pd.DataFrame(historia)[["epoch", "eval_loss", "eval_mae"]]

In [ ]:
p_va = modelos.predecir_bert(modelo, tok, list(va.texto_final), TOPE, MAX_LEN, HP["estrategia"])
torch.cuda.synchronize(); t0 = time.perf_counter()
p_te = modelos.predecir_bert(modelo, tok, list(te.texto_final), TOPE, MAX_LEN, HP["estrategia"])
torch.cuda.synchronize(); t_inf = (time.perf_counter() - t0) / len(te)
p_te_ref = modelos.predecir_bert(modelo, tok, list(te.texto_ref), TOPE, MAX_LEN, HP["estrategia"])
pd.concat([pd.DataFrame({"id": va.id, "split": "val", "y": va.y, "pred": p_va}),
           pd.DataFrame({"id": te.id, "split": "test", "y": te.y, "pred": p_te, "pred_ref": p_te_ref})]
          ).to_csv(PRED / "bert.csv", index=False)
print(f"MAE val {mean_absolute_error(va.y, p_va):.3f}")

(config.TABLAS / "bert_config.json").write_text(json.dumps({
    **HP, "tope": TOPE, "escala_target": f"y / {TOPE}", "hardware": GPU,
    "transformers": transformers.__version__, "torch": torch.__version__,
    "segundos_entrenamiento": seg_entrenamiento, "segundos_inferencia_por_cv": t_inf,
    "epocas_corridas": len(historia), "mejor_mae_val": min(h["eval_mae"] for h in historia)}, indent=2))
(config.TABLAS / "tiempos_bert.json").write_text(json.dumps({"bert": [seg_entrenamiento, t_inf]}))

## Pregunta 3 · Sensibilidad de BERT

Una decisión a la vez, mismo split y semilla (≈ 10 min cada una en T4):

- truncamiento `cabeza` vs `cabeza_cola`;
- pérdida `huber` vs `mse`.

In [ ]:
sens = [{"modelo": "bert", "decision": "base", "variante": "cabeza_cola + mse",
         "MAE_test": mean_absolute_error(te.y, p_te)}]
for decision, cambio in [("truncamiento", {"estrategia": "cabeza"}), ("perdida", {"perdida": "huber"})]:
    hp = {**HP, **cambio}
    m, t, seg, _ = modelos.entrenar_bert(list(tr.texto_final), tr.y.values, list(va.texto_final), va.y.values,
                                         TOPE, config.MODELOS / f"bert_{decision}", guardar=False, **hp)
    p = modelos.predecir_bert(m, t, list(te.texto_final), TOPE, MAX_LEN, hp["estrategia"])
    sens.append({"modelo": "bert", "decision": decision, "variante": f"{hp['estrategia']} + {hp['perdida']}",
                 "MAE_test": mean_absolute_error(te.y, p), "min_entrenamiento": seg / 60})
    del m; torch.cuda.empty_cache()
sens = pd.DataFrame(sens)
sens.to_csv(config.TABLAS / "sensibilidad_bert.csv", index=False)
sens.round(3)

Los pesos quedan en `MyDrive/HW_04_202602/models/bert_regresor/` (**no** van al repo). Para la
app, descarga esa carpeta a `models/bert_regresor/` de tu repo local.

In [ ]:
import shutil
if EN_COLAB:
    shutil.make_archive("/content/entregables_04", "zip", RAIZ, "output")
    # Solo lo que usa la app (sin las variantes de sensibilidad): ~270 MB
    app = Path("/content/modelos_para_app/models"); shutil.rmtree(app.parent, ignore_errors=True); app.mkdir(parents=True)
    shutil.copytree(config.MODELOS / "bert_regresor", app / "bert_regresor")
    for f in ("tfidf_ridge.joblib", "embeddings_ridge.joblib", "mediana.json", "regla.json"):
        shutil.copy2(config.MODELOS / f, app / f)
    shutil.make_archive("/content/modelos_para_app", "zip", app.parent)
    from google.colab import files
    files.download("/content/entregables_04.zip")
    files.download("/content/modelos_para_app.zip")